In [0]:
%python
# Databricks notebook source
# ============================================================================
# RetailPulse | M1 - Bronze Ingestion with Auto Loader
# ----------------------------------------------------------------------------
# Purpose:
#   Ingest all landing datasets into Bronze incrementally (exactly-once),
#   preserving raw data as-is with ingestion metadata.
#
# Design:
#   - Source      : /Volumes/workspace/rp_landing/raw/landing/<dataset>
#   - Bronze      : workspace.rp_bronze.<table>
#   - Checkpoints : /Volumes/workspace/rp_landing/ops/chk/<table>
#   - Schemas     : /Volumes/workspace/rp_landing/ops/schemas/<dataset>
#   - Trigger     : availableNow (Free Edition compatible, incremental-batch)
#   - Bronze rule : Append-only. No cleaning, no deduplication.
#                   Only adds _ingest_ts and _source_file.
#
# Usage:
#   Run for a single dataset:
#       dbutils.widgets.text("dataset", "sales_daily")
#   Or run all datasets from the driver cell at the bottom.
# ============================================================================

from pyspark.sql import functions as F

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------

CATALOG = "workspace"
LANDING_ROOT = f"/Volumes/{CATALOG}/rp_landing/raw/landing"
OPS_ROOT = f"/Volumes/{CATALOG}/rp_landing/ops"

# Dataset -> (landing folder, bronze table, source format, schema evolution mode)
# schemaEvolutionMode = "addNewColumns" is used so that drift columns
# (e.g. promo_flag in week 5, channel in week 10) are added automatically
# and unparseable fields are preserved in _rescued_data.
DATASETS = {
    "sales_daily": {
        "folder": "sales_daily",
        "table": "sales_daily",
        "format": "json",
        "evolution_mode": "addNewColumns",
    },
    "stores": {
        "folder": "stores",
        "table": "stores",
        "format": "csv",
        "evolution_mode": "addNewColumns",
    },
    "products": {
        "folder": "products",
        "table": "products",
        "format": "csv",
        "evolution_mode": "addNewColumns",
    },
    "sales_history": {
        "folder": "sales_history",
        "table": "sales_history",
        "format": "csv",
        "evolution_mode": "addNewColumns",
    },
    "forecast_runs": {
        "folder": "forecast_runs",
        "table": "forecast_runs",
        "format": "csv",
        "evolution_mode": "addNewColumns",
    },
    "planner_overrides": {
        "folder": "planner_overrides",
        "table": "planner_overrides",
        "format": "json",
        "evolution_mode": "addNewColumns",
    },
}

# ----------------------------------------------------------------------------
# Ingestion Function
# ----------------------------------------------------------------------------

def ingest_dataset(dataset_key: str) -> None:
    """
    Ingest one dataset from Landing to Bronze using Auto Loader.

    Args:
        dataset_key: Key from DATASETS dict (e.g. "sales_daily").

    Behaviour:
        - Reads with format("cloudFiles") + trigger(availableNow=True).
        - Writes append-only to rp_bronze.<table>.
        - If no new files exist, no rows are written (idempotent re-run).
    """
    config = DATASETS[dataset_key]
    source_path = f"{LANDING_ROOT}/{config['folder']}"
    bronze_table = f"{CATALOG}.rp_bronze.{config['table']}"
    checkpoint_path = f"{OPS_ROOT}/chk/bronze_{config['table']}"
    schema_path = f"{OPS_ROOT}/schemas/{config['folder']}"

    print(f"Ingesting {dataset_key}: {source_path} -> {bronze_table}")

    reader = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", config["format"])
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", config["evolution_mode"])
        .option("cloudFiles.inferColumnTypes", "true")
    )

    # CSV sources require header handling; JSON sources do not.
    if config["format"] == "csv":
        reader = reader.option("header", "true")

    stream = (
        reader.load(source_path)
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingest_ts"),
        )
    )

    query = (
        stream.writeStream
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(bronze_table)
    )
    query.awaitTermination()
    print(f"Completed: {dataset_key} -> {bronze_table}")

# ----------------------------------------------------------------------------
# Driver - Run All Datasets
# ----------------------------------------------------------------------------

def run_all() -> None:
    """Ingest all configured datasets in a fixed, deterministic order."""
    for dataset_key in DATASETS:
        ingest_dataset(dataset_key)

# Uncomment to run all datasets from this notebook:
# run_all()

# To run M1 datasets (only these 3 have files in landing right now):
ingest_dataset("sales_daily")
ingest_dataset("stores")
ingest_dataset("products")

In [0]:
SELECT max(sale_date) as last_day, count(*) as total_rows 
FROM workspace.rp_bronze.sales_daily

In [0]:
%python
for t in ["stores","products","sales_history","forecast_runs","sales","planner_overrides"]:
    print(t, spark.table(f"workspace.rp_bronze.{t}").count())

In [0]:
%python
ingest_dataset("sales_history")
ingest_dataset("forecast_runs")
ingest_dataset("planner_overrides")

In [0]:
SELECT 'sales_daily' as table_name, COUNT(*) as cnt FROM workspace.rp_bronze.sales_daily
UNION ALL
SELECT 'stores', COUNT(*) FROM workspace.rp_bronze.stores
UNION ALL
SELECT 'products', COUNT(*) FROM workspace.rp_bronze.products
UNION ALL
SELECT 'sales_history', COUNT(*) FROM workspace.rp_bronze.sales_history
UNION ALL
SELECT 'forecast_runs', COUNT(*) FROM workspace.rp_bronze.forecast_runs
UNION ALL
SELECT 'planner_overrides', COUNT(*) FROM workspace.rp_bronze.planner_overrides

In [0]:
SELECT COUNT(*) as rescued_rows FROM workspace.rp_bronze.sales_daily WHERE _rescued_data IS NOT NULL

In [0]:
DESCRIBE TABLE workspace.rp_bronze.sales_daily

In [0]:
CREATE TABLE IF NOT EXISTS workspace.rp_bronze.stores_copyinto
(
  store_id STRING,
  banner STRING,
  region STRING,
  format STRING,
  status STRING,
  _source_file STRING,
  _ingest_ts TIMESTAMP
)
USING DELTA

In [0]:
COPY INTO workspace.rp_bronze.stores_copyinto (store_id, banner, region, format, status)
FROM '/Volumes/workspace/rp_landing/raw/landing/stores'
FILEFORMAT = JSON

In [0]:
DROP TABLE IF EXISTS workspace.rp_bronze.stores_copyinto;

CREATE TABLE workspace.rp_bronze.stores_copyinto
(
  store_id STRING,
  banner STRING,
  region STRING,
  format STRING,
  status STRING
)
USING DELTA

In [0]:
COPY INTO workspace.rp_bronze.stores_copyinto
FROM '/Volumes/workspace/rp_landing/raw/landing/stores'
FILEFORMAT = JSON

In [0]:
%python
import os
base = "/Volumes/workspace/rp_landing/raw/landing/stores"
for root, dirs, files in os.walk(base):
    for f in files[:5]:
        print(os.path.join(root, f))
    if files:
        break

In [0]:
COPY INTO workspace.rp_bronze.stores_copyinto
FROM '/Volumes/workspace/rp_landing/raw/landing/stores'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
SELECT 'auto_loader' AS source, COUNT(*) AS cnt FROM workspace.rp_bronze.stores
UNION ALL
SELECT 'copy_into' AS source, COUNT(*) AS cnt FROM workspace.rp_bronze.stores_copyinto

In [0]:
COPY INTO workspace.rp_bronze.stores_copyinto
FROM '/Volumes/workspace/rp_landing/raw/landing/stores'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
SELECT table_name FROM workspace.information_schema.tables
WHERE table_schema = 'rp_bronze'
ORDER BY table_name

In [0]:
SELECT 'stores' AS table_name, COUNT(*) AS cnt FROM workspace.rp_bronze.stores
UNION ALL SELECT 'products', COUNT(*) FROM workspace.rp_bronze.products
UNION ALL SELECT 'sales_history', COUNT(*) FROM workspace.rp_bronze.sales_history
UNION ALL SELECT 'sales', COUNT(*) FROM workspace.rp_bronze.sales
UNION ALL SELECT 'forecast_runs', COUNT(*) FROM workspace.rp_bronze.forecast_runs
UNION ALL SELECT 'planner_overrides', COUNT(*) FROM workspace.rp_bronze.planner_overrides

In [0]:
SELECT COUNT(*) AS rescued_rows FROM workspace.rp_bronze.sales WHERE _rescued_data IS NOT NULL

In [0]:
DESCRIBE TABLE workspace.rp_bronze.sales

In [0]:
SELECT MAX(sale_date) AS last_sale_date, COUNT(*) AS total_rows
FROM workspace.rp_bronze.sales